<img src="https://upload.wikimedia.org/wikipedia/commons/8/89/TheNorthFace_logo.svg" alt="THE NORTH FACE LOGO" width="50%" />

# The North Face e-commerce : boosting online sales !

## Company's Description 📇

The North Face is an American outdoor recreation product company, founded in 1968 to supply climbers. The North Face produces clothing, footwear, and outdoor equipment. By the late 1990s customers had expanded beyond outdoor enthusiasts and in the 2000s it became a fashion style symbol.


## Project 🚧

The marketing department would like to take advantage of machine learning solutions to boost online sales on the website : https://www.thenorthface.fr/

They have identified two major solutions that could have a huge effect on the conversion rates :
- Deploying a **recommender system** that will allow to suggest additionnal products to users, that are similar to the items they are already interested in. The recommendations could be materialized by a "you might also be interested by these products..." section that would appear on each product page of the website.
- Improving the structure of the products catalog thanks to **topic extraction**. The idea is to use unsupervised methods to challenge the existing categories : is it possible to find new categories of product that would be more suitable for the navigation on the website ?

## Goals 🎯

The project can be cut into three steps :

1. Identify groups of products that have similar descriptions.

2. Use the groups of similar products to build a simple recommender system algorithm.

3. Use topic modeling algorithms to automatically assess the latent topics present in the item descriptions.

## Scope of this project 🖼️

In this project, you'll work with a corpus of item descriptions from The North Face's product catalog. Data can be found here: 

👉👉<a href="https://www.kaggle.com/cclark/product-item-data?select=sample-data.csv" target="_blank"> The North Face product catalog</a> 👈👈


## Helpers 🦮

To help you achieve this project, here are a few tips that should help you: 

### Preprocessing of textual data

- First, you'll have to make some preprocessings to clean the corpus (in particular, don't forget to handle stop words and to lemmatize the documents). For this step, you'll need `pandas` and `spacy`.

- Then, you can to encode the texts with **TF-IDF transformation** (this can be done with scikit learn's `TFIDFVectorizer`)

### Part 1 : Groups of products with similar descriptions

For this part, you'll have to train a **clustering model** that will create groups of products for which the descriptions are "close" to each other. 
- You can use `DBSCAN` to make some clustering on the TF-IDF matrix. When dealing with texts, the distance metric to be used is "cosine" instead of "euclidean". You have to find some values for `eps` and `min_samples` that allow to get 10-20 clusters and not too many outliers.
- You can display a `wordcloud` for each cluster to analyze the results and see if the groups contain different words.

### Part 2 - Recommender system

Then, you can use the cluster ids from part 1 to build a **recommender system**. The aim is to be able to suggest to a user some products that are similar to the ones he is interested in. To do this, we will consider that products belonging to the same cluster are similar.

- Create a function named `find_similar_items` that takes an argument `item_id` representing the id of a product, and that returns a list of 5 item ids that belong to the same cluster as the product passed in argument
- Use python's `input()` function to allow the user to choose a product and perform some suggestions of similar items

### Part 3 : Topic modeling

*This part is independant from the two others.*

The aim is to use an LSA model to automatically extract latent topics in the products' descriptions.

- Use TruncatedSVD to make some topic extraction from the TF-IDF matrix. You can try values for `n_components` that allow to get 10-20 topics. Save the encoded matrix into a variable named `topic_encoded_df`.
- Contrary to clustering, LSA allows to map each document to a mixing of several topics. For this reason, it's a bit more difficult to interpret the topics as categories : one document can actually be related to several topics at a time. To make things easier, consider extracting the main topic of each document.
- As in part 1, you can display `wordclouds` to analyze the results.

## Deliverable 📬

To complete this project, your team should: 

- Train at least one **clustering model** on the corpus and display wordclouds describing the clusters
- Develop a simple python code that allows a user to type the id of a product he's interested in, and then get a list of similar items
- Train at least one **TruncatedSVD model** on the corpus and display wordclouds describing the latent topics


# 0. Load and clean data 
### Preprocessing of textual data

- First, you'll have to make some preprocessings to clean the corpus (in particular, don't forget to handle stop words and to lemmatize the documents). For this step, you'll need `pandas` and `spacy`.

- Then, you can to encode the texts with **TF-IDF transformation** (this can be done with scikit learn's `TFIDFVectorizer`)

In [41]:
# Load data 
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("cclark/product-item-data")

# print("Path to dataset files:", path)




In [42]:
import pandas as pd
import spacy

#data = path
dataset_init = pd.read_csv("sample-data.csv",encoding='utf-8')
dataset_init.head()

,id,description
0,1,Active classic boxers - There's a reason why o...
1,2,Active sport boxer briefs - Skinning up Glory ...
2,3,Active sport briefs - These superbreathable no...
3,4,"Alpine guide pants - Skin in, climb ice, switc..."
4,5,"Alpine wind jkt - On high ridges, steep ice an..."


In [43]:
dataset_init["description_"] = dataset_init["description"]

In [44]:
dataset = dataset_init.copy()

In [ ]:

dataset["name"] = dataset["description"].astype("string").str.split(" - ", n=1).str[0].str.strip()
dataset["description"] = dataset["description"].astype("string").str.split(" - ", n=1).str[1].str.strip()
dataset["category"] = dataset["name"].astype("string").str.split(" ",n=1).str[0].str.strip()
dataset["article"] = dataset["name"].astype("string").str.split(" ",n=1).str[1].str.strip()
#Details
dataset["Details"] = dataset["description"].astype("string").str.split('Details:</b><ul> <li>',n=1).str[1]
#Fabric
dataset.head()

,id,description,description_,name,category,article
0,1,There's a reason why our boxers are a cult fav...,Active classic boxers - There's a reason why o...,Active classic boxers,Active,classic boxers
1,2,Skinning up Glory requires enough movement wit...,Active sport boxer briefs - Skinning up Glory ...,Active sport boxer briefs,Active,sport boxer briefs
2,3,These superbreathable no-fly briefs are the mi...,Active sport briefs - These superbreathable no...,Active sport briefs,Active,sport briefs
3,4,"Skin in, climb ice, switch to rock, traverse a...","Alpine guide pants - Skin in, climb ice, switc...",Alpine guide pants,Alpine,guide pants
4,5,"On high ridges, steep ice and anything alpine,...","Alpine wind jkt - On high ridges, steep ice an...",Alpine wind jkt,Alpine,wind jkt


In [ ]:
dico_words = {
    "jkt":"jacket",
    "in." : "",
    "fz"   : "",
    "b.c." : "",
    "btm" : "",
    "S/s" : "Short-Sleeved",
    "R1"  : "",
    "Lw"  : "Lightweight",
    "Mw"  : "" 
    "R2"  : "",
    "R1"  : "" , 
    "r4" : "" , 
    "Mlc" : "",
    "L/s" : "",
    "M10" : "",
    
    
    
    
    
    
        
    
}

In [46]:
dataset.sample(12)

,id,description,description_,name,category,article
446,447,"Akin to the North Wind, Caribou cover over 3,0...",Caribou north wind t-shirt - Akin to the North...,Caribou north wind t-shirt,Caribou,north wind t-shirt
388,389,This graceful rendition of the courier bag is ...,Lw travel courier - This graceful rendition of...,Lw travel courier,Lw,travel courier
181,182,The Elsa Tunic flourishes in heat-prone latitu...,Elsa tunic - The Elsa Tunic flourishes in heat...,Elsa tunic,Elsa,tunic
411,412,"This ultralight, moisture wicking fishing shir...","Sunshade shirt - This ultralight, moisture wic...",Sunshade shirt,Sunshade,shirt
431,432,Soft-wearing ringspun organic cotton; Screen-p...,Girl's live simply deer t-shirt - Soft-wearing...,Girl's live simply deer t-shirt,Girl's,live simply deer t-shirt
308,309,For corn skiing and other transition-season en...,Aravis 1/4 zip - For corn skiing and other tra...,Aravis 1/4 zip,Aravis,1/4 zip
5,6,Our most technical soft shell for full-on moun...,Ascensionist jkt - Our most technical soft she...,Ascensionist jkt,Ascensionist,jkt
10,11,"Soft, stretchy polyester fabric is fast wickin...","Baby sunshade top - Soft, stretchy polyester f...",Baby sunshade top,Baby,sunshade top
342,343,"First and foremost, these lightweight, soft, s...","Rock guide pants - First and foremost, these l...",Rock guide pants,Rock,guide pants
304,305,reg - You'll like the flattering drape of thes...,Inter-continental pants - reg - You'll like th...,Inter-continental pants,Inter-continental,pants


In [60]:
#import plus musclé avec Claude Code : 
import html
import re
import pandas as pd

RE_NOM     = re.compile(r"^\s*(.*?)\s-\s(?=[A-Z0-9'])", re.DOTALL)
RE_INTRO   = re.compile(r"^(.*?)(?:<br><br><b>Details:</b>|$)", re.DOTALL)
RE_DETAIL  = re.compile(r"<li>(.*?)</li>", re.DOTALL)
RE_TISSU   = re.compile(r"<b>Fabric:\s*</b>(.*?)(?:<br><br>|$)", re.DOTALL)
# Accepte "(340 g 11.8 oz)" et "99 g (3.5 oz)"
RE_POIDS   = re.compile(r"<b>Weight:\s*</b>\(?\s*(\d+(?:\.\d+)?)\s*g\s*\(?\s*(\d+(?:\.\d+)?)\s*oz\s*\)")
RE_ORIGINE = re.compile(r"(?:Made|Printed)\s+in\s+([A-Za-z][A-Za-z .]*?)\.?\s*$")

def separer_nom(desc: str, ecart_max: int = 15) -> tuple[str | None, str]:
    """Renvoie (nom, reste). Garde le 2e ' - ' dans le nom s'il est à <= ecart_max caractères du 1er."""
    d = desc.strip()
    pos = [m.start() for m in re.finditer(r" - ", d)]
    if not pos:
        return None, d
    fin = pos[1] if len(pos) > 1 and pos[1] - pos[0] <= ecart_max else pos[0]
    return d[:fin].strip(), d[fin + 3:].strip()

def _nettoyer(s: str) -> str:
    """Retire les guillemets englobants, dédouble les guillemets et décode les entités HTML."""
    s = s.strip()
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
        s = s[1:-1]
    return html.unescape(s.replace('""', '"')).strip()


def parser_description(desc) -> dict:
    vide = {"name": None, "text": None, "details": [], "fabric": None,
            "weight_g": None, "weight_oz": None, "origin": None}
    if not isinstance(desc, str) or not desc.strip():
        return vide
    desc = desc.strip()
    # m_nom = RE_NOM.match(desc)
    # reste = desc[m_nom.end():] if m_nom else desc
    name, reste = separer_nom(desc)
    
    m_poids, m_tissu, m_orig = RE_POIDS.search(reste), RE_TISSU.search(reste), RE_ORIGINE.search(reste)
    return {
        "name": name,
        "text": _nettoyer(RE_INTRO.match(reste).group(1)).replace('""', '"'),
        
        "details": [_nettoyer(x) for x in RE_DETAIL.findall(reste)],
        "fabric": _nettoyer(m_tissu.group(1)) if m_tissu else None,
        "weight_g": float(m_poids.group(1)) if m_poids else None,
        "weight_oz": float(m_poids.group(2)) if m_poids else None,
        "origin": m_orig.group(1).strip() if m_orig else None,
    }


def charger_produits(chemin: str) -> pd.DataFrame:
    brut = pd.read_csv(chemin, header=0, names=["id", "description"], encoding="utf-8")
    parsed = pd.DataFrame([parser_description(x) for x in brut["description"]], index=brut.index)
    out = pd.concat([brut[["id"]], parsed], axis=1)
    # Les posters n'ont pas de poids : on ne signale que les autres champs
    manquants = out[out[["name", "fabric", "origin"]].isna().any(axis=1)]
    if not manquants.empty:
        print(f"{len(manquants)} ligne(s) incomplète(s) : id = {manquants['id'].tolist()}")
    return out


dataset = charger_produits("sample-data.csv")
dataset["details_txt"] = dataset["details"].str.join(" | ")   # export CSV

In [61]:
dataset

,id,name,text,details,fabric,weight_g,weight_oz,origin,details_txt
0,1,Active classic boxers,There's a reason why our boxers are a cult fav...,"[Silky Capilene 1 fabric is ultralight, breath...",3.7-oz 100% all-recycled polyester with Gladio...,99.0,3.5,Mexico,"Silky Capilene 1 fabric is ultralight, breatha..."
1,2,Active sport boxer briefs,Skinning up Glory requires enough movement wit...,"[Stretch mesh provides support, open-weave mes...",4.6-oz 93% polyester (71% recycled)/7% spandex...,60.0,2.1,Israel,"Stretch mesh provides support, open-weave mesh..."
2,3,Active sport briefs,These superbreathable no-fly briefs are the mi...,"[Stretch mesh provides support, open-weave mes...",4.6-oz 93% polyester (71% recycled)/7% spandex...,49.0,1.7,Israel,"Stretch mesh provides support, open-weave mesh..."
3,4,Alpine guide pants,"Skin in, climb ice, switch to rock, traverse a...",[Durable stretch-woven polyester (47% recycled...,7.6-oz 90-denier 92% polyester (47% recycled)/...,597.0,20.7,Vietnam,Durable stretch-woven polyester (47% recycled)...
4,5,Alpine wind jkt,"On high ridges, steep ice and anything alpine,...","[Lightweight, breathable polyester ripstop fab...",Shell: 1.3-oz 20-denier 100% polyester ripstop...,331.0,11.5,China,"Lightweight, breathable polyester ripstop fabr..."
...,...,...,...,...,...,...,...,...,...
495,496,Cap 2 bottoms,Cut loose from the maddening crowds and search...,[Capilene 2 fabric has an open knit that invit...,4-oz 100% polyester (54% recycled) with Gladio...,155.0,5.5,Mexico,Capilene 2 fabric has an open knit that invite...
496,497,Cap 2 crew,This crew takes the edge off fickle weather. I...,[Capilene 2 fabric has an open knit that invit...,4-oz 100% polyester (54% recycled) with Gladio...,170.0,6.0,Mexico,Capilene 2 fabric has an open knit that invite...
497,498,All-time shell,No need to use that morning Times as an umbrel...,[2.5-layer 100% all-recycled polyester shell w...,"Shell: 2.5-layer, 4.1-oz 50-denier 100% all-re...",493.0,17.4,China,2.5-layer 100% all-recycled polyester shell wi...
498,499,All-wear cargo shorts,All-Wear Cargo Shorts bask in the glory of swe...,"[Lightweight organic cotton canvas, Zip fly wi...",5.2-oz 100% organic cotton canvas. Recyclable ...,349.0,12.1,Thailand,Lightweight organic cotton canvas | Zip fly wi...


In [ ]:
#je vais tokenizer ce datraframe, quelle est la meilleure strategie : juste text ou toutes les colonnes


Ça dépend de ta tâche, mais en général : un seul texte composite (name, text, details, fabric), avec les colonnes numériques et catégorielles gardées à part.

Ce que je recommande

Tokenizer : name, text, details et fabric. Ils portent le sens du produit. Mets name en premier, pour qu’une troncature ne le coupe jamais.
Ne pas tokenizer weight_g, weight_oz et origin : ce sont des features structurées (numérique, catégorielle). Dans un tokenizer, 340 devient des sous-mots sans valeur numérique.
Retirer le boilerplate : « Recyclable through the Common Threads Recycling Program » est dans fabric de 351 lignes sur 500. Ça gaspille des tokens sans information.
Éviter la fuite de cible : si tu prédis origin ou le poids, exclus-les du texte. Ici origin est de toute façon dans la description d’origine (« Made in … »).
Dédoublonner avant le split : 66 noms sont dupliqués (même produit, variantes de couleur). Sinon train et test partagent des quasi-doublons.

In [ ]:


import re
import pandas as pd

BOILERPLATE = re.compile(r"\s*Recyclable through (?:the|our) Common Threads Recycling Program\.?", re.I)


def construire_texte(df: pd.DataFrame, avec_details: bool = True, avec_tissu: bool = True) -> pd.Series:
    """Texte composite par produit : nom d'abord (protégé de la troncature), puis le reste."""
    requis = {"name", "text", "details", "fabric"}
    if not requis <= set(df.columns):
        raise KeyError(f"Colonnes manquantes : {sorted(requis - set(df.columns))}")

    def un_produit(r) -> str:
        parts = [str(r["name"] or "").strip(), str(r["text"] or "").strip()]
        if avec_details and isinstance(r["details"], list):
            parts.append(" ; ".join(r["details"]))
        if avec_tissu and isinstance(r["fabric"], str):
            parts.append("Tissu : " + r["fabric"])
        txt = ". ".join(p.rstrip(". ") for p in parts if p)
        return re.sub(r"\s+", " ", BOILERPLATE.sub("", txt)).strip()

    return df.apply(un_produit, axis=1)


def mesurer_tokens(textes: pd.Series, tokenizer) -> pd.Series:
    """Longueurs en tokens (sans troncature) pour choisir max_length."""
    n = [len(tokenizer(t, add_special_tokens=True)["input_ids"]) for t in textes]
    print(pd.Series(n).describe(percentiles=[.5, .9, .95, .99]).round(0).to_string())
    return pd.Series(n, index=textes.index)


# Usage (Windows / Apple Silicon / CUDA : le tokenizer tourne sur CPU, pas de souci)
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")   # ou le modèle que tu vas utiliser
dataset["texte"] = construire_texte(dataset)
dataset = dataset.drop_duplicates(subset="texte").reset_index(drop=True)   # avant le split !

longueurs = mesurer_tokens(dataset["texte"], tok)
enc = tok(dataset["texte"].tolist(), truncation=True, max_length=int(longueurs.quantile(0.95)),
          padding=True, return_tensors="pt")

features a cote du texte 

In [ ]:
import numpy as np

X_num = dataset[["weight_g"]].assign(weight_g=lambda d: d["weight_g"].fillna(d["weight_g"].median()))
X_cat = pd.get_dummies(dataset["origin"], prefix="orig")      # ou OrdinalEncoder

In [ ]:
# pretraitement avec les abreviations :


Valeurs que j’ai ajoutées : tu n’avais renseigné que jkt, in., S/s, Lw. Ce sont mes propositions, à valider : btm, ctn, p/o, pkt, fz, reg, b.c., L/s, Mw, Hw, Ulw, pfd. J’ai trouvé ces abréviations dans les noms de tes produits.

Clés laissées vides : R1, R2, r4, M10 et Mlc sont des noms de gamme (R1 = polaire Regulator, par exemple), pas des abréviations. Les remplacer par un mot supprimerait une information utile au clustering, donc je les ignore par défaut. rapport_abreviations indique qu’elles apparaissent 43, 10, 6, 6 et 7 fois. Si tu veux les expliciter, mets par exemple "R1": "R1 fleece".

Autres abréviations non traitées : a/c (3 occurrences) et Sst (1) ont un sens que je ne devine pas.

Résultats vérifiés :

Les 6 noms avec in. donnent par exemple Minimalist board shorts-19 inch.
Dans l’id 31, 15" to 17"" donne bien 15 inch to 17 inch.
Dans l’id 110, 30"; regular, 32"" donne 30 inch ; regular, 32 inch.

Pour la lemmatisation qui suit : t.is_alpha écarte les nombres, donc 19 inch devient inch. Si la taille compte, garde t.like_num. Autre piège : Shop pants - regular devient bien regular, mais un mot comme Regular isolé sera aussi traité, ce qui est voulu.

In [ ]:

import re
import pandas as pd

# Abréviation -> mot complet. Valeur "" = pas encore défini : la clé est ignorée (voir rapport_abreviations)
DICO_MOTS = {
    "jkt": "jacket", "btm": "bottom", "ctn": "cotton", "p/o": "pullover", "pkt": "pocket",
    "fz": "full-zip", "reg": "regular", "b.c.": "British Columbia",
    "S/s": "short sleeve", "L/s": "long sleeve",
    "Lw": "lightweight", "Mw": "midweight", "Hw": "heavyweight", "Ulw": "ultralight",
    "pfd": "personal flotation device",
    "R1": "", "R2": "", "r4": "", "M10": "", "Mlc": "",
}

# Cas contextuels : "in." seulement après un chiffre ; " (pouce) seulement après un chiffre ou une fraction
SPECIAUX = [
    (re.compile(r"(?<=\d)\s*in\.(?!\w)", re.I), " inch"),
    (re.compile(r"(?<=[\d/½¼¾])\s*\"+"), " inch"),
]


def compiler_dico(dico: dict) -> list[tuple[re.Pattern, str]]:
    """Regex par clé (insensible à la casse, mot entier), clés longues d'abord. Ignore les valeurs vides."""
    regs = []
    for cle in sorted(dico, key=len, reverse=True):
        if not dico[cle].strip():
            continue
        regs.append((re.compile(rf"(?<![\w/]){re.escape(cle)}(?![\w/])", re.I), dico[cle]))
    return regs


def remplacer_abreviations(texte, regs):
    if not isinstance(texte, str):
        return texte
    for pat, rep in SPECIAUX + regs:
        texte = pat.sub(rep, texte)
    return re.sub(r"\s+", " ", texte).strip()


def appliquer_dico(df: pd.DataFrame, dico: dict = DICO_MOTS, cols=("name", "text", "details", "fabric")) -> pd.DataFrame:
    """Copie de df avec abréviations remplacées (colonnes texte ou listes de textes)."""
    regs = compiler_dico(dico)
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            raise KeyError(f"Colonne absente : {c}")
        out[c] = out[c].map(lambda v: [remplacer_abreviations(x, regs) for x in v] if isinstance(v, list)
                            else remplacer_abreviations(v, regs))
    return out


def rapport_abreviations(df: pd.DataFrame, dico: dict = DICO_MOTS, cols=("name", "text", "details", "fabric")) -> pd.DataFrame:
    """Occurrences de chaque clé du dico dans le corpus, et si elle est définie."""
    textes = pd.concat([df[c].explode() if df[c].map(lambda v: isinstance(v, list)).any() else df[c]
                        for c in cols]).dropna().astype(str)
    lignes = []
    for cle, val in dico.items():
        pat = re.compile(rf"(?<![\w/]){re.escape(cle)}(?![\w/])", re.I)
        lignes.append({"cle": cle, "remplacement": val or "(non défini)", "occurrences": int(textes.str.count(pat).sum())})
    return pd.DataFrame(lignes).sort_values("occurrences", ascending=False).reset_index(drop=True)


# Usage : AVANT construire_texte et la lemmatisation
print(rapport_abreviations(dataset))
dataset = appliquer_dico(dataset)

je veux lemmatize le texte et ensuite TFIDF transformation avec tfidfVectorizer pour train un clustering model et regrouper les produits aux descriptions similaires

Pipeline en 4 étapes : lemmatisation spaCy, TfidfVectorizer, réduction LSA (TruncatedSVD) puis KMeans. Le texte vient de construire_texte (boilerplate déjà retiré).

Points d’attention :

Silhouette faible : sur du texte TF-IDF, 0,05 à 0,2 est normal. Juge surtout la cohérence des termes_par_cluster et des noms regroupés.
Chiffres supprimés : t.is_alpha écarte les nombres (3.7-oz, Cap 3), donc « Cap 1/2/3/4 » sont proches. Pour les distinguer, garde les tokens alphanumériques (t.is_alpha or t.like_num), ou ajoute le poids comme feature numérique à côté.
Vocabulaire métier : si un terme domine tous les clusters (par exemple fabric, oz), passe-le dans mots_exclus=frozenset({...}), ou baisse max_df.
Fit sur tout le jeu : le clustering est non supervisé, donc ajuster le TF-IDF sur tous les produits est correct. Si tu évalues ensuite un modèle supervisé, ajuste-le sur le train uniquement.
Alternatives : AgglomerativeClustering(metric="cosine", linkage="average") ne demande pas de choisir k à l’avance si tu fixes distance_threshold. HDBSCAN (sklearn.cluster.HDBSCAN) détecte aussi les produits « isolés ».
Compatibilité : tout tourne sur CPU, sous Windows comme sur Apple Silicon. Aucun tenseur torch n’est utilisé.

In [ ]:
# pip install spacy scikit-learn
# python -m spacy download en_core_web_sm      (descriptions en anglais)
import numpy as np
import pandas as pd
import spacy
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import Normalizer


def charger_spacy(modele: str = "en_core_web_sm"):
    """Charge spaCy sans parser ni NER (inutiles ici, donc plus rapide)."""
    try:
        return spacy.load(modele, disable=["parser", "ner"])
    except OSError as e:
        raise OSError(f"Modèle absent. Lance : python -m spacy download {modele}") from e


def lemmatiser(textes, nlp, mots_exclus: frozenset = frozenset()) -> list[str]:
    """Lemmes alphabétiques en minuscules, sans stop words ni mots exclus."""
    sortie = []
    # n_process=1 : le multiprocessing de spaCy est fragile sous Windows / notebook
    for doc in nlp.pipe((str(t) for t in textes), batch_size=64):
        lemmes = [t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop and len(t) > 1]
        sortie.append(" ".join(l for l in lemmes if l not in mots_exclus))
    return sortie


def vectoriser(textes_lem, min_df=2, max_df=0.7, ngram=(1, 2)):
    """TF-IDF (norme L2, sous-linéaire) sur les textes déjà lemmatisés."""
    vec = TfidfVectorizer(ngram_range=ngram, min_df=min_df, max_df=max_df,
                          sublinear_tf=True, token_pattern=r"(?u)\b[a-z][a-z]+\b")
    X = vec.fit_transform(textes_lem)
    vide = int((X.getnnz(axis=1) == 0).sum())
    if vide:
        print(f"⚠️ {vide} texte(s) sans aucun terme retenu (vecteur nul)")
    return vec, X


def reduire_lsa(X, n_comp=100, seed=42):
    """LSA + normalisation L2 : KMeans euclidien ≈ distance cosinus."""
    n = max(2, min(n_comp, X.shape[1] - 1, X.shape[0] - 1))
    Z = TruncatedSVD(n_components=n, random_state=seed).fit_transform(X)
    return Normalizer(copy=False).fit_transform(Z)


def choisir_k(Z, ks=range(5, 31, 5), seed=42) -> tuple[int, pd.Series]:
    """Meilleur k selon le silhouette (cosinus)."""
    scores = {}
    for k in ks:
        if k >= len(Z):
            break
        lab = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(Z)
        scores[k] = silhouette_score(Z, lab, metric="cosine")
    s = pd.Series(scores, name="silhouette")
    return int(s.idxmax()), s


def termes_par_cluster(X, labels, vec, top=8) -> pd.DataFrame:
    """Termes TF-IDF les plus forts du centroïde de chaque cluster."""
    termes = np.array(vec.get_feature_names_out())
    lignes = []
    for k in np.unique(labels):
        centre = np.asarray(X[labels == k].mean(axis=0)).ravel()
        lignes.append({"cluster": k, "n": int((labels == k).sum()),
                       "termes": ", ".join(termes[np.argsort(-centre)[:top]])})
    return pd.DataFrame(lignes).set_index("cluster")


# ---------- Usage ----------
nlp = charger_spacy()
dataset["texte"] = construire_texte(dataset)
dataset["texte_lem"] = lemmatiser(dataset["texte"], nlp)

vec, X = vectoriser(dataset["texte_lem"])
Z = reduire_lsa(X)

k, scores = choisir_k(Z)
print(scores.round(3).to_string(), f"\n-> k = {k}")

dataset["cluster"] = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Z)
print(termes_par_cluster(X, dataset["cluster"].to_numpy(), vec).to_string())
print(dataset.groupby("cluster")["name"].apply(lambda s: ", ".join(s.head(5))).to_string())


# Bonus : produits les plus proches d'un produit donné (cosinus)
def voisins(i: int, Z, df, n=5) -> pd.DataFrame:
    sim = Z @ Z[i]                       # Z est normalisée : produit scalaire = cosinus
    idx = np.argsort(-sim)[1:n + 1]
    return df.iloc[idx][["id", "name"]].assign(similarite=sim[idx].round(3))

In [47]:
import spacy
spacy.__version__

'3.8.16'

In [48]:
!python -m spacy download en_core_news_sm 


✘ No compatible package found for 'en_core_news_sm' (spaCy v3.8.16)



In [49]:
!python -m spacy download en_core_web_sm 

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 8.5 MB/s eta 0:00:02
     -------- ------------------------------- 2.6/12.8 MB 8.9 MB/s eta 0:00:02
     ---------------- ----------------------- 5.2/12.8 MB 10.3 MB/s eta 0:00:01
     ------------------------- -------------- 8.1/12.8 MB 11.2 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 11.3 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 11.5 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [50]:
# let's create a complete corpus : 
# Import english stop words
from spacy.lang.en.stop_words import STOP_WORDS

In [51]:
print(STOP_WORDS)

{'she', 'mostly', 'but', 'side', 'two', 'may', 'above', 'move', 'along', '’re', 'once', 'anyway', 'quite', 'around', 'first', 'each', 'show', 're', '’ve', 'sometimes', 'through', 'every', 'thence', 'thru', 'latterly', 'anywhere', 'twelve', 'of', 'last', 'might', 'via', 'mine', 'done', 'again', 'itself', 'under', 'unless', 'upon', 'former', 'go', 'him', 'beforehand', 'another', 'your', 'into', 'hereafter', 'back', '‘s', 'both', 'per', 'thereby', 'various', 'off', 'least', 'twenty', 'serious', 'thereupon', 'whereupon', 'namely', 'other', 'becomes', 'still', 'also', 'they', '‘re', 'any', 'does', "'m", "'ve", 'full', 'between', 'cannot', 'forty', '‘m', 'the', 'hence', 'out', 'keep', 'thereafter', 'nobody', 'ever', 'same', 'herein', 'we', 'wherein', 'four', 'very', 'would', 'anyhow', 'most', 'regarding', 'from', 'had', 'together', 'up', 'even', 'until', 'amount', 'whence', '‘ve', 'yourself', 'therefore', 'either', "n't", 'yourselves', 'therein', 'about', 'if', 'put', 'eight', 'not', 'top', 

In [ ]:
# 500 articles with Id  | text : nom article - description 
# Baggies shorts - Even Baggies, our most popular shorts for anything, or nothing, occasionally need an update. This season we've increased the inseam length. Their casual fit, quick-drying water-repellent nylon and elasticized waistband with an internal drawstring remain the same as ever. Other features include a polyester mesh lining, a rear snap pocket and front pockets (with self-draining mesh corners) positioned to reduce drag in the water. Inseam (size M) is 7"". Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>Quick-drying nylon with a DWR (durable water repellent) finish</li> <li>Elasticized waistband with internal drawstring; black mesh liner</li> <li>Vertical on-seam side pockets for reduced drag in the water; pocket bags have quick-drain-and-dry mesh corners; snap-closed back pocket</li> <li>""Inseam is 7""""""</li></ul><br><br><b>Fabric: </b>4.2-oz 100% nylon with a DWR finish. Lining: 5.2-oz 100% polyester mesh. Recyclable through the Common Threads Recycling Program<br><br><b>Weight: </b>(233 g 8.1 oz)<br><br>Made in Nicaragua."
# Tokenizer : name, text, details et fabric
# lowercase caracters of description
dataset["description_clean"] = (
    dataset['description']
    .str.lower()
    .str.replace(r"[^a-z0-9]+"," ",regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
dataset.head()

,id,description,description_,name,category,article,description_clean
0,1,There's a reason why our boxers are a cult fav...,Active classic boxers - There's a reason why o...,Active classic boxers,Active,classic boxers,there s a reason why our boxers are a cult fav...
1,2,Skinning up Glory requires enough movement wit...,Active sport boxer briefs - Skinning up Glory ...,Active sport boxer briefs,Active,sport boxer briefs,skinning up glory requires enough movement wit...
2,3,These superbreathable no-fly briefs are the mi...,Active sport briefs - These superbreathable no...,Active sport briefs,Active,sport briefs,these superbreathable no fly briefs are the mi...
3,4,"Skin in, climb ice, switch to rock, traverse a...","Alpine guide pants - Skin in, climb ice, switc...",Alpine guide pants,Alpine,guide pants,skin in climb ice switch to rock traverse a kn...
4,5,"On high ridges, steep ice and anything alpine,...","Alpine wind jkt - On high ridges, steep ice an...",Alpine wind jkt,Alpine,wind jkt,on high ridges steep ice and anything alpine t...


In [53]:
# create object nlp:
import en_core_web_sm
nlp = en_core_web_sm.load()

In [54]:
# tokenize: 
# Tokenize the cleaned sentences and remove english stopwords
from spacy.lang.en.stop_words import STOP_WORDS
dataset['description_tokenized'] = (
    dataset['description_clean']
    .apply(lambda x: nlp(x))
    .apply(lambda doc: [token.lemma_ for token in doc if token.text not in STOP_WORDS])
)
dataset.head()

,id,description,description_,name,category,article,description_clean,description_tokenized
0,1,There's a reason why our boxers are a cult fav...,Active classic boxers - There's a reason why o...,Active classic boxers,Active,classic boxers,there s a reason why our boxers are a cult fav...,"[s, reason, boxer, cult, favorite, cool, espec..."
1,2,Skinning up Glory requires enough movement wit...,Active sport boxer briefs - Skinning up Glory ...,Active sport boxer briefs,Active,sport boxer briefs,skinning up glory requires enough movement wit...,"[skin, glory, require, movement, boxer, decide..."
2,3,These superbreathable no-fly briefs are the mi...,Active sport briefs - These superbreathable no...,Active sport briefs,Active,sport briefs,these superbreathable no fly briefs are the mi...,"[superbreathable, fly, brief, minimalist, s, c..."
3,4,"Skin in, climb ice, switch to rock, traverse a...","Alpine guide pants - Skin in, climb ice, switc...",Alpine guide pants,Alpine,guide pants,skin in climb ice switch to rock traverse a kn...,"[skin, climb, ice, switch, rock, traverse, kni..."
4,5,"On high ridges, steep ice and anything alpine,...","Alpine wind jkt - On high ridges, steep ice an...",Alpine wind jkt,Alpine,wind jkt,on high ridges steep ice and anything alpine t...,"[high, ridge, steep, ice, alpine, jacket, serv..."


In [55]:
# Detokenize the tokenized sentences and store them in an ```nlp_ready``` column
dataset['nlp_ready']= dataset['description_tokenized'].apply(lambda x: ' '.join(x))
dataset.head()

,id,description,description_,name,category,article,description_clean,description_tokenized,nlp_ready
0,1,There's a reason why our boxers are a cult fav...,Active classic boxers - There's a reason why o...,Active classic boxers,Active,classic boxers,there s a reason why our boxers are a cult fav...,"[s, reason, boxer, cult, favorite, cool, espec...",s reason boxer cult favorite cool especially s...
1,2,Skinning up Glory requires enough movement wit...,Active sport boxer briefs - Skinning up Glory ...,Active sport boxer briefs,Active,sport boxer briefs,skinning up glory requires enough movement wit...,"[skin, glory, require, movement, boxer, decide...",skin glory require movement boxer decide poach...
2,3,These superbreathable no-fly briefs are the mi...,Active sport briefs - These superbreathable no...,Active sport briefs,Active,sport briefs,these superbreathable no fly briefs are the mi...,"[superbreathable, fly, brief, minimalist, s, c...",superbreathable fly brief minimalist s choice ...
3,4,"Skin in, climb ice, switch to rock, traverse a...","Alpine guide pants - Skin in, climb ice, switc...",Alpine guide pants,Alpine,guide pants,skin in climb ice switch to rock traverse a kn...,"[skin, climb, ice, switch, rock, traverse, kni...",skin climb ice switch rock traverse knife edge...
4,5,"On high ridges, steep ice and anything alpine,...","Alpine wind jkt - On high ridges, steep ice an...",Alpine wind jkt,Alpine,wind jkt,on high ridges steep ice and anything alpine t...,"[high, ridge, steep, ice, alpine, jacket, serv...",high ridge steep ice alpine jacket serve true ...


In [56]:
#Use sklearn to calculate the tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy

# TruncatedSVD from sklearn
from sklearn.decomposition import TruncatedSVD

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(dataset['nlp_ready'])
dense = X.toarray()
dense

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(500, 3807))

In [57]:
vectorizer.vocabulary_

{'reason': 2656,
 'boxer': 623,
 'cult': 1006,
 'favorite': 1372,
 'cool': 932,
 'especially': 1301,
 'sticky': 3175,
 'situation': 2998,
 'quick': 2613,
 'dry': 1191,
 'lightweight': 1994,
 'underwear': 3549,
 'minimal': 2167,
 'space': 3100,
 'travel': 3468,
 'pack': 2370,
 'expose': 1332,
 'brush': 654,
 'waistband': 3661,
 'offer': 2309,
 'skin': 3010,
 'softness': 3080,
 'panel': 2388,
 'construction': 910,
 'traditional': 3451,
 'classic': 798,
 'fit': 1415,
 'functional': 1517,
 'fly': 1457,
 'oz': 2367,
 '100': 3,
 'recycled': 2666,
 'polyester': 2511,
 'moisture': 2190,
 'wicke': 3737,
 'performance': 2441,
 'inseam': 1823,
 'size': 2999,
 'recyclable': 2664,
 'common': 866,
 'thread': 3389,
 'recycle': 2665,
 'program': 2567,
 'br': 625,
 'details': 1098,
 'ul': 3525,
 'li': 1983,
 'silky': 2977,
 'capilene': 709,
 'fabric': 1349,
 'ultralight': 3530,
 'breathable': 632,
 'elastic': 1235,
 'comfort': 857,
 'gladiodor': 1559,
 'natural': 2237,
 'odor': 2308,
 'control': 923,
 

# Part 1 : Groups of products with similar descriptions

For this part, you'll have to train a **clustering model** that will create groups of products for which the descriptions are "close" to each other. 
- You can use `DBSCAN` to make some clustering on the TF-IDF matrix. When dealing with texts, the distance metric to be used is "cosine" instead of "euclidean". You have to find some values for `eps` and `min_samples` that allow to get 10-20 clusters and not too many outliers.
- You can display a `wordcloud` for each cluster to analyze the results and see if the groups contain different words.
